# Waste2Worth

This is the **first version** of ML model for Waste2Worth:

1. **Waste Classification** : predicts waste category (and hazard level) from a listing's text description
2. **Value & Environmental Impact Estimation** : estimates market value, disposal cost savings, and CO₂ reduction
3. **Buyer Recommendation** : content-based matching of listings to buyer interests

Since real platform data doesn't exist yet, it generates a mock dataset that mimics what real Waste2Worth listings would look like, then trains model on it.


In [13]:
import pandas as pd
import numpy as np
import random
import re
import joblib
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## 1. Synthetic Dataset Generation

We define a set of realistic waste categories (matching common industrial waste types), then generate:
- A **listings dataset** (description, category, hazard level, quantity, condition, location) — used to train the classifier
- Realistic **price/value data** per listing — used to train the valuation model
- A small **buyer profile dataset** — used to demo the recommendation engine


In [14]:
# ---- Waste category taxonomy ----
# category -> (hazard_level, base_price_per_kg_usd, co2_savings_per_kg_kg, typical descriptors)
CATEGORY_INFO = {
    "Metal Scrap":        ("Low",      0.35, 1.8, ["steel offcuts", "aluminum shavings", "scrap metal sheets", "copper wire ends", "iron filings"]),
    "Plastic Waste":      ("Low",      0.20, 1.2, ["HDPE offcuts", "PET bottle scrap", "plastic packaging waste", "polypropylene granules", "mixed plastic trims"]),
    "Textile Waste":      ("Low",      0.15, 0.9, ["fabric offcuts", "cotton scrap", "denim trimmings", "yarn waste", "dye-stained textile rolls"]),
    "Organic Waste":      ("Low",      0.05, 0.4, ["food processing residue", "fruit pulp waste", "vegetable trimmings", "spent grain", "agricultural byproduct"]),
    "Wood Waste":         ("Low",      0.10, 0.7, ["sawdust", "wood offcuts", "pallet scrap", "plywood trimmings", "timber shavings"]),
    "Chemical Residue":   ("High",     0.40, 2.1, ["solvent residue", "spent catalyst", "chemical sludge", "resin waste", "industrial coolant residue"]),
    "Construction Debris":("Medium",  0.08, 0.6, ["concrete rubble", "brick waste", "drywall offcuts", "tile fragments", "mixed construction debris"]),
    "E-Waste":            ("High",    0.60, 2.5, ["circuit board scrap", "cable waste", "used batteries", "electronic component offcuts", "damaged PCB units"]),
    "Paper & Cardboard":  ("Low",     0.12, 0.5, ["cardboard offcuts", "paper trimmings", "packaging paper waste", "corrugated scrap", "shredded paper waste"]),
    "Rubber Waste":       ("Medium",  0.18, 1.0, ["tire scrap", "rubber sheet offcuts", "conveyor belt waste", "rubber gasket trimmings", "vulcanized rubber scrap"]),
}

CONDITIONS = ["Clean / sorted", "Mixed / unsorted", "Contaminated", "Baled", "Loose"]
LOCATIONS = ["Delhi NCR", "Mumbai", "Pune", "Chennai", "Bengaluru", "Ahmedabad", "Hyderabad", "Kolkata"]

def make_description(category):
    _, _, _, descriptors = CATEGORY_INFO[category]
    desc = random.choice(descriptors)
    qty_word = random.choice(["large batch of", "regular supply of", "surplus of", "steady stream of", "one-time lot of"])
    return f"{qty_word} {desc} generated from our production line, available for pickup"

def generate_listings(n=600):
    rows = []
    categories = list(CATEGORY_INFO.keys())
    for i in range(n):
        category = random.choice(categories)
        hazard, base_price, co2_factor, _ = CATEGORY_INFO[category]
        condition = random.choice(CONDITIONS)
        quantity_kg = round(np.random.gamma(shape=2.0, scale=800), 1)  # skewed realistic quantities
        location = random.choice(LOCATIONS)
        description = make_description(category)

        # price varies with condition + noise
        condition_multiplier = {"Clean / sorted": 1.15, "Baled": 1.05, "Mixed / unsorted": 0.85,
                                 "Loose": 0.95, "Contaminated": 0.55}[condition]
        noise = np.random.normal(1.0, 0.1)
        price_per_kg = max(0.01, base_price * condition_multiplier * noise)
        market_value = round(price_per_kg * quantity_kg, 2)
        disposal_cost_saved = round(quantity_kg * 0.06, 2)  # assume $0.06/kg avoided landfill fee
        co2_reduction_kg = round(quantity_kg * co2_factor * condition_multiplier, 1)

        rows.append({
            "listing_id": f"L{i:04d}",
            "description": description,
            "category": category,
            "hazard_level": hazard,
            "condition": condition,
            "quantity_kg": quantity_kg,
            "location": location,
            "price_per_kg": round(price_per_kg, 3),
            "market_value_usd": market_value,
            "disposal_cost_saved_usd": disposal_cost_saved,
            "co2_reduction_kg": co2_reduction_kg,
        })
    return pd.DataFrame(rows)

listings_df = generate_listings(600)
listings_df.head()


,listing_id,description,category,hazard_level,condition,quantity_kg,location,price_per_kg,market_value_usd,disposal_cost_saved_usd,co2_reduction_kg
0,L0000,regular supply of PET bottle scrap generated f...,Plastic Waste,Low,Clean / sorted,1914.9,Bengaluru,0.227,434.34,114.89,2642.6
1,L0001,steady stream of dye-stained textile rolls gen...,Textile Waste,Low,Clean / sorted,483.1,Mumbai,0.178,85.99,28.99,500.0
2,L0002,regular supply of aluminum shavings generated ...,Metal Scrap,Low,Clean / sorted,3719.8,Mumbai,0.433,1612.12,223.19,7700.0
3,L0003,regular supply of shredded paper waste generat...,Paper & Cardboard,Low,Loose,818.9,Delhi NCR,0.108,88.45,49.13,389.0
4,L0004,one-time lot of corrugated scrap generated fro...,Paper & Cardboard,Low,Baled,1599.2,Chennai,0.102,162.95,95.95,839.6


In [15]:
# Quick sanity check on class balance and value ranges
print(listings_df['category'].value_counts())
print()
print(listings_df[['quantity_kg', 'market_value_usd', 'co2_reduction_kg']].describe())


category
Construction Debris    77
E-Waste                70
Metal Scrap            63
Wood Waste             63
Textile Waste          62
Paper & Cardboard      56
Organic Waste          55
Rubber Waste           55
Chemical Residue       55
Plastic Waste          44
Name: count, dtype: int64

       quantity_kg  market_value_usd  co2_reduction_kg
count   600.000000        600.000000        600.000000
mean   1659.780333        348.831883       1810.781000
std    1152.274580        429.387315       1959.751147
min      19.800000          1.590000         12.500000
25%     805.175000         90.632500        581.275000
50%    1396.300000        201.785000       1161.300000
75%    2175.825000        420.380000       2203.925000
max    6753.400000       3181.230000      13932.300000


## 2. Waste Classification Model (Text-Based)

We train a baseline **TF-IDF + Logistic Regression** classifier to predict `category` from the listing `description`.

Hazard level is derived from the predicted category via a lookup table (rule-based) rather than predicted directly — this keeps the system safe and predictable for a compliance-relevant field, and is a legitimate design choice to call out in your report.


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    listings_df["description"], listings_df["category"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=listings_df["category"]
)

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_vec = tfidf_vectorizer.fit_transform(X_train)
X_test_vec = tfidf_vectorizer.transform(X_test)

classifier = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
classifier.fit(X_train_vec, y_train)

y_pred = classifier.predict(X_test_vec)
print(classification_report(y_test, y_pred))


                     precision    recall  f1-score   support

   Chemical Residue       1.00      1.00      1.00        11
Construction Debris       1.00      1.00      1.00        15
            E-Waste       1.00      1.00      1.00        14
        Metal Scrap       1.00      1.00      1.00        13
      Organic Waste       1.00      1.00      1.00        11
  Paper & Cardboard       1.00      1.00      1.00        11
      Plastic Waste       1.00      1.00      1.00         9
       Rubber Waste       1.00      1.00      1.00        11
      Textile Waste       1.00      1.00      1.00        12
         Wood Waste       1.00      1.00      1.00        13

           accuracy                           1.00       120
          macro avg       1.00      1.00      1.00       120
       weighted avg       1.00      1.00      1.00       120



In [17]:
def predict_category(description: str):
    """Predict waste category + hazard level from a free-text description."""
    vec = tfidf_vectorizer.transform([description])
    category = classifier.predict(vec)[0]
    confidence = float(np.max(classifier.predict_proba(vec)))
    hazard_level = CATEGORY_INFO[category][0]
    return {
        "category": category,
        "hazard_level": hazard_level,
        "confidence": round(confidence, 3),
    }

# quick test
predict_category("large batch of steel offcuts generated from our production line, available for pickup")


{'category': 'Metal Scrap', 'hazard_level': 'Low', 'confidence': 0.484}

## 3. Value & Environmental Impact Estimation

Two layers here:
- A **rule-based estimator** (category + condition lookup) — simple, transparent, works from day one with zero transaction history
- A **regression model** (Random Forest) trained on the synthetic price data — shows the ML technique and is a drop-in upgrade path once real transaction data accumulates


In [18]:
# ---- Rule-based estimator ----
def estimate_value_rule_based(category: str, condition: str, quantity_kg: float):
    base_price, co2_factor = CATEGORY_INFO[category][1], CATEGORY_INFO[category][2]
    condition_multiplier = {"Clean / sorted": 1.15, "Baled": 1.05, "Mixed / unsorted": 0.85,
                             "Loose": 0.95, "Contaminated": 0.55}.get(condition, 0.9)
    price_per_kg = base_price * condition_multiplier
    market_value = round(price_per_kg * quantity_kg, 2)
    disposal_cost_saved = round(quantity_kg * 0.06, 2)
    co2_reduction_kg = round(quantity_kg * co2_factor * condition_multiplier, 1)
    return {
        "estimated_value_usd": market_value,
        "disposal_cost_saved_usd": disposal_cost_saved,
        "co2_reduction_kg": co2_reduction_kg,
    }

estimate_value_rule_based("Metal Scrap", "Clean / sorted", 1000)


{'estimated_value_usd': 402.5,
 'disposal_cost_saved_usd': 60.0,
 'co2_reduction_kg': 2070.0}

In [19]:
# ---- ML regression estimator (Random Forest) ----
features_df = pd.get_dummies(listings_df[["category", "condition", "quantity_kg"]],
                              columns=["category", "condition"])
target = listings_df["market_value_usd"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    features_df, target, test_size=0.2, random_state=RANDOM_STATE
)

value_model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
value_model.fit(Xr_train, yr_train)

preds = value_model.predict(Xr_test)
print("MAE (USD):", round(mean_absolute_error(yr_test, preds), 2))

value_model_columns = features_df.columns.tolist()  # save for inference-time alignment


MAE (USD): 38.56


In [20]:
def estimate_value_ml(category: str, condition: str, quantity_kg: float):
    """Predict market value using the trained Random Forest regressor."""
    row = {col: 0 for col in value_model_columns}
    row["quantity_kg"] = quantity_kg
    cat_col = f"category_{category}"
    cond_col = f"condition_{condition}"
    if cat_col in row: row[cat_col] = 1
    if cond_col in row: row[cond_col] = 1
    X_input = pd.DataFrame([row])[value_model_columns]
    predicted_value = float(value_model.predict(X_input)[0])
    return {"estimated_value_usd": round(predicted_value, 2)}

estimate_value_ml("Metal Scrap", "Clean / sorted", 1000)


{'estimated_value_usd': 385.87}

## 4. Buyer Recommendation (Content-Based)

Since there's no bidding/transaction history yet (cold start), we use **content-based filtering**: represent each listing and each buyer's stated material interests as TF-IDF vectors over the same vocabulary, then rank listings for a buyer by cosine similarity.

Once real bid/transaction data exists, this can be extended with collaborative filtering (e.g., using the `surprise` library) layered on top.


In [21]:
# ---- Synthetic buyer profiles ----
BUYER_PROFILES = {
    "buyer_steelworks_co": "we need steel offcuts and scrap metal sheets for remelting",
    "buyer_ecoplastics":   "looking for HDPE and PET plastic packaging waste for recycling",
    "buyer_textile_mill":  "sourcing fabric offcuts and cotton scrap for fiber reprocessing",
    "buyer_agrifeed":      "interested in food processing residue and vegetable trimmings for animal feed",
    "buyer_greenboard":    "need wood offcuts and sawdust for particle board manufacturing",
}

# Fit a shared TF-IDF space across listing descriptions
rec_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
listing_vectors = rec_vectorizer.fit_transform(listings_df["description"])

def recommend_listings_for_buyer(buyer_key: str, top_n=5):
    buyer_text = BUYER_PROFILES[buyer_key]
    buyer_vec = rec_vectorizer.transform([buyer_text])
    similarities = cosine_similarity(buyer_vec, listing_vectors).flatten()
    top_idx = similarities.argsort()[::-1][:top_n]
    results = listings_df.iloc[top_idx][["listing_id", "category", "description", "market_value_usd"]].copy()
    results["match_score"] = similarities[top_idx].round(3)
    return results.reset_index(drop=True)

recommend_listings_for_buyer("buyer_steelworks_co")


,listing_id,category,description,market_value_usd,match_score
0,L0522,Metal Scrap,surplus of scrap metal sheets generated from o...,322.17,0.588
1,L0264,Metal Scrap,surplus of scrap metal sheets generated from o...,436.85,0.588
2,L0202,Metal Scrap,steady stream of scrap metal sheets generated ...,104.99,0.571
3,L0293,Metal Scrap,steady stream of scrap metal sheets generated ...,323.60,0.571
4,L0403,Metal Scrap,steady stream of scrap metal sheets generated ...,617.11,0.571


## 5. Combined Pipeline Function

This is the single function you'd call from your backend when a **new listing is submitted**: it classifies the waste, estimates its value/impact, and returns a clean JSON-serializable result — matching FR-3 and FR-4 from the SRS in one call.


In [22]:
def classify_and_value(description: str, condition: str, quantity_kg: float, use_ml_valuation: bool = False):
    """
    End-to-end function for a new listing submission.
    Call this from your backend (FastAPI/Flask/Django view) right after a listing is created.
    """
    classification = predict_category(description)
    category = classification["category"]

    if use_ml_valuation:
        valuation = estimate_value_ml(category, condition, quantity_kg)
        # ML regressor only predicts value; still get CO2/disposal savings from rule-based logic
        rule_extras = estimate_value_rule_based(category, condition, quantity_kg)
        valuation["disposal_cost_saved_usd"] = rule_extras["disposal_cost_saved_usd"]
        valuation["co2_reduction_kg"] = rule_extras["co2_reduction_kg"]
    else:
        valuation = estimate_value_rule_based(category, condition, quantity_kg)

    return {
        "category": category,
        "hazard_level": classification["hazard_level"],
        "classification_confidence": classification["confidence"],
        **valuation,
    }

# Example: this is what you'd send back to the frontend as the "AI Analysis" section of a listing
classify_and_value(
    description="surplus of copper wire ends generated from our production line, available for pickup",
    condition="Clean / sorted",
    quantity_kg=500,
)


{'category': 'Metal Scrap',
 'hazard_level': 'Low',
 'classification_confidence': 0.628,
 'estimated_value_usd': 201.25,
 'disposal_cost_saved_usd': 30.0,
 'co2_reduction_kg': 1035.0}

## 6. Saving Models for Integration


In [23]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(tfidf_vectorizer, "models/tfidf_vectorizer.pkl")
joblib.dump(classifier, "models/category_classifier.pkl")
joblib.dump(value_model, "models/value_regressor.pkl")
joblib.dump(value_model_columns, "models/value_model_columns.pkl")
joblib.dump(rec_vectorizer, "models/recommendation_vectorizer.pkl")
joblib.dump(CATEGORY_INFO, "models/category_info.pkl")

listings_df.to_csv("models/synthetic_listings.csv", index=False)

print("Saved to ./models/:")
for f in sorted(os.listdir("models")):
    print(" -", f)


Saved to ./models/:
 - category_classifier.pkl
 - category_info.pkl
 - recommendation_vectorizer.pkl
 - synthetic_listings.csv
 - tfidf_vectorizer.pkl
 - value_model_columns.pkl
 - value_regressor.pkl


In [24]:
# ---- Example: how to reload the models in your backend service ----
loaded_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
loaded_classifier = joblib.load("models/category_classifier.pkl")
loaded_category_info = joblib.load("models/category_info.pkl")

def predict_category_from_saved(description: str):
    vec = loaded_vectorizer.transform([description])
    category = loaded_classifier.predict(vec)[0]
    hazard_level = loaded_category_info[category][0]
    return {"category": category, "hazard_level": hazard_level}

predict_category_from_saved("mixed plastic trims from our packaging line")


{'category': 'Plastic Waste', 'hazard_level': 'Low'}